In [ ]:
!pip install -q git+https://github.com/joyennn/CONQUER.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from conquer import *

set_api_key("your-api-key")

In [ ]:
##### 1. Corpus Processing #####

corpus = dp("corpus.txt")

CONQUER Parser
Existing parsed corpus found.
Loading      : /content/parsed/corpus.parquet


In [ ]:
corpus = load_parsed("parsed/corpus.parquet")

In [ ]:
corpus.preview(15)

,id,text,lemma,upos,head,deprel
249,1,Every,every,DET,2,det
250,2,guy,guy,NOUN,4,nmod:poss
251,3,'s,'s,PART,2,case
252,4,nephews,nephew,NOUN,5,nsubj
253,5,research,research,VERB,0,root
254,6,three,three,NUM,7,nummod
255,7,closets,closet,NOUN,5,obj
256,8,and,and,CCONJ,11,cc
257,9,teenagers,teenager,NOUN,11,nmod:poss
258,10,','s,PART,9,case


In [ ]:
##### 2. Natural Language Query Planning #####
plan1 = plan_query("One token has lemma by.")
plan2 = plan_query("The ROOT token has an aux:pass dependent.")
plan3 = plan_query("The ROOT token has an obl:agent dependent.")


passive_construction = plan_set(
    name="passive",
    plans=[plan1, plan2, plan3],
)


In [ ]:
##### 3. Query Plan #####
show(passive_construction)
save(passive_construction, "plans/passive.json")

{
  "type": "plan_set",
  "name": "passive",
  "description": "",
  "mode": "and",
  "target": "sentence",
  "plans": [
    {
      "target": "sentence",
      "description": "Sentences containing a token with lemma 'by'.",
      "include": [
        {
          "name": "by_lemma",
          "type": "token_condition",
          "field": "lemma",
          "operator": "equals",
          "value": "by",
          "list_file": null,
          "head": null,
          "head_field": null,
          "head_operator": null,
          "head_value": null
        }
      ],
      "exclude": [],
      "notes": [],
      "type": "plan"
    },
    {
      "target": "sentence",
      "description": "Find sentences where the ROOT token has an aux:pass dependent.",
      "include": [
        {
          "name": "root_with_passive_aux",
          "type": "dependency_condition",
          "field": "deprel",
          "operator": "equals",
          "value": "aux:pass",
          "list_file": null,
       

In [ ]:
##### 4. Validation #####

report = validate(passive_construction)
show_report(report)

Validation Report
Status          : VALID
Object type     : plan_set
Normalized type : plan
Include rules   : 4
Exclude rules   : 0
------------------------------------------------------------
Errors:
  None

Warnings:
  None

Suggestions:
  None


In [ ]:
##### 4. Compilation #####

compiled = compile_plan(corpus, passive_construction)
show_code(compiled)

# --------------------------------------------------
# CONQUER Generated Python Query
# --------------------------------------------------
# Description: 

# Input:
#   sent_df = one sentence-level slice of the parsed corpus

# Include conditions
# ------------------
# condition1: token_condition: lemma equals by
condition1 = (
    (sent_df["lemma"].astype(str) == 'by').any()
)

# condition2: dependency_condition: deprel equals aux:pass; head=root
condition2 = (
    (
        dep_rows := sent_df.loc[(sent_df["deprel"].astype(str) == 'aux:pass')]
    )
    # and at least one dependent has a matching head
    head condition: sent_df.loc[sent_df["id"] == dep["head"], "deprel"] == "root"
)

# condition3: token_condition: deprel equals root
condition3 = (
    (sent_df["deprel"].astype(str) == 'root').any()
)

# condition4: dependency_condition: deprel equals obl:agent; head=root
condition4 = (
    (
        dep_rows := sent_df.loc[(sent_df["deprel"].astype(str) == 'obl:agent')]
    )
    # 

In [ ]:
##### 5. Query Execution #####

results = apply(corpus, compiled)

In [ ]:
##### 6. Result Management #####

results.summary()

{'matched_sentences': 1961,
 'matched_tokens': 16991,
 'avg_sentence_length': 8.664456909739929,
 'top_lemmas': {'by': 1968,
  'be': 1962,
  '.': 1961,
  'not': 990,
  'some': 697,
  'the': 691,
  "'s": 508,
  'of': 219,
  'a': 184,
  'that': 182},
 'top_upos': {'NOUN': 2916,
  'ADP': 2522,
  'DET': 2100,
  'AUX': 1962,
  'PUNCT': 1961,
  'VERB': 1958,
  'PROPN': 1838,
  'PART': 1501,
  'ADJ': 213,
  'ADV': 20},
 'top_deprels': {'case': 2713,
  'det': 2100,
  'obl:agent': 1962,
  'aux:pass': 1962,
  'nsubj:pass': 1962,
  'punct': 1961,
  'root': 1961,
  'advmod': 1010,
  'nmod:poss': 508,
  'obl': 267}}

In [ ]:
results.preview()

,sent_id,sentence
0,12000,Lucille's sisters are confused by Amy.
1,12001,A lot of hospitals were astounded by some actr...
2,12002,Larry's oncologists aren't helped by Spain.
3,12003,Jeffrey's sons are insulted by Tina's supervisor.
4,12004,Tracy isn't fired by Jodi's daughter.
5,12005,Eva's bosses weren't cared for by drivers.
6,12006,Some guys are disgusted by Angela.
7,12007,That guest's supervisor wasn't disliked by Way...
8,12008,Curtis was distracted by these high schools.
9,12009,David wasn't disgusted by these hospitals.


In [ ]:
results.df

,sent_id,sentence
0,12000,Lucille's sisters are confused by Amy.
1,12001,A lot of hospitals were astounded by some actr...
2,12002,Larry's oncologists aren't helped by Spain.
3,12003,Jeffrey's sons are insulted by Tina's supervisor.
4,12004,Tracy isn't fired by Jodi's daughter.
...,...,...
1956,30995,This pork was taken by some guy.
1957,30996,Newspaper articles weren't scanned by the men.
1958,30997,Liam is talked about by the dancers.
1959,30998,Monet is sounded like by the man.


In [ ]:
results.tokens()

,sent_id,sentence,id,text,lemma,upos,xpos,feats,head,deprel,start_char,end_char
111316,12000,Lucille's sisters are confused by Amy.,1,Lucille,Lucille,PROPN,NNP,Number=Sing,3,nmod:poss,77082,77089
111317,12000,Lucille's sisters are confused by Amy.,2,'s,'s,PART,POS,None,1,case,77089,77091
111318,12000,Lucille's sisters are confused by Amy.,3,sisters,sister,NOUN,NNS,Number=Plur,5,nsubj:pass,77092,77099
111319,12000,Lucille's sisters are confused by Amy.,4,are,be,AUX,VBP,Mood=Ind|Number=Plur|Person=3|Tense=Pres|VerbF...,5,aux:pass,77100,77103
111320,12000,Lucille's sisters are confused by Amy.,5,confused,confuse,VERB,VBN,Tense=Past|VerbForm=Part|Voice=Pass,0,root,77104,77112
...,...,...,...,...,...,...,...,...,...,...,...,...
274267,30999,Paul was kissed by some girl.,3,kissed,kiss,VERB,VBN,Tense=Past|VerbForm=Part|Voice=Pass,0,root,40867,40873
274268,30999,Paul was kissed by some girl.,4,by,by,ADP,IN,None,6,case,40874,40876
274269,30999,Paul was kissed by some girl.,5,some,some,DET,DT,PronType=Ind,6,det,40877,40881
274270,30999,Paul was kissed by some girl.,6,girl,girl,NOUN,NN,Number=Sing,3,obl:agent,40882,40886


In [ ]:
results.save_csv("outputs/passive_construction_sentences.csv")
results.save_tokens_csv("outputs/passive_construction_tokens.csv")
results.save_txt("outputs/passive_construction_sentences.txt")